In [34]:
import os
import csv
import ast
import re
import time
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import accuracy_score, hamming_loss, precision_recall_fscore_support
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier

In [3]:
# Variables globales:
SEED = 2627
DEV_PART = 0.15
CORPUS_DIR = "data"
TRAIN_FILE = os.path.join(CORPUS_DIR, "dataset_train.csv")
TEST_FILE = os.path.join(CORPUS_DIR, "dataset_test.csv")
TOKEN_PATTERN = r"\S+" # Captura cualquier grupo de caracteres que NO contenga espacios

In [4]:
# Función para cargar el corpus:
def load_corpus(filename):
    papers, terms = [], []

    # Abrimos el archivo 'filename' y lo leemos línea por línea:
    with open(filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=',')

        # Saltamos la fila con los títulos de las columnas:
        next(reader)

        # Extraemos las columnas del archivo:
        for row in reader:
            if not row or len(row) < 3:
                continue

            papers.append((row[0], row[1]))
            terms.append(ast.literal_eval(row[2]))

    return papers, terms


# Función de tokenización:
def tokenize(text):
    # Buscamos palabras que empiecen por letras y puedan contener guiones
    # (para evitar ruido como "5k" y no excluir estructuras como "state-of-the-art"):
    pattern = re.compile(r"(?u)\b[a-zA-Z][\w-]+\b")
    return " ".join(pattern.findall(text))


# Función para calcular las métricas de evaluación:
def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    hl = hamming_loss(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl,
    }

In [21]:
# Cargamos el corpus, y guardamos las entradas y los términos:
raw_all, y_all = load_corpus(TRAIN_FILE)
print("Papers de entrenamiento:", len(raw_all))

# Convertimos los términos en listas binarias:
mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y_all)

# Creamos una partición aleatoria de entrenamiento/desarrollo:
raw_train, raw_dev, y_train, y_dev = train_test_split(raw_all, y_bin, test_size=DEV_PART, shuffle=True, random_state=SEED)

# Tokenizamos los títulos y los resúmenes:
train_tok = [tokenize(title) + tokenize(summary) for title, summary in raw_train]
dev_tok = [tokenize(title) + tokenize(summary) for title, summary in raw_dev]

Papers de entrenamiento: 43555


In [38]:
print(compute_metrics(y_dev, np.zeros_like(y_dev)))
print(compute_metrics(y_dev, np.ones_like(y_dev)))

{'accuracy': 0.0, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0, 'hamming_loss': 0.06210918710918711}
{'accuracy': 0.0, 'f1': 0.08888139346828376, 'precision': 0.062109187109187114, 'recall': 1.0, 'hamming_loss': 0.9378908128908129}


In [ ]:
vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True)
A = vec.fit_transform(train_tok)
B = vec.transform(dev_tok)

multilabel = {}
for c in [0.1, 0.25, 0.5, 0.75, 1, 3, 5, 6, 7, 8, 9, 10]:
    multilabel[c] = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A, y_train)
    y_pred = multilabel[c].predict(B)
    print(f"=== C={c}")
    print(compute_metrics(y_dev, y_pred))

=== C=0.1, 0.1
{'accuracy': 0.25068870523415976, 'f1': 0.3828775210093977, 'precision': 0.31286728597454694, 'recall': 0.6942435956382225, 'hamming_loss': 0.046564126109580654}
=== C=0.25, 0.25
{'accuracy': 0.3111417202326293, 'f1': 0.39915850688501014, 'precision': 0.33524251235804814, 'recall': 0.6494154681800712, 'hamming_loss': 0.04026192662556299}
=== C=0.5, 0.5
{'accuracy': 0.37327823691460055, 'f1': 0.43541858424590946, 'precision': 0.38825915061601834, 'recall': 0.6411783595984153, 'hamming_loss': 0.03542459224277406}
=== C=0.75, 0.75
{'accuracy': 0.4097030915212733, 'f1': 0.4547737544094469, 'precision': 0.41272095939201936, 'recall': 0.6301159019055236, 'hamming_loss': 0.0327517600244873}
=== C=1, 1
{'accuracy': 0.43357820630547905, 'f1': 0.4639510064438256, 'precision': 0.4297905105335521, 'recall': 0.6102667049626519, 'hamming_loss': 0.030969871878962787}
=== C=3, 3
{'accuracy': 0.5238751147842057, 'f1': 0.5138451094128209, 'precision': 0.5954641341570015, 'recall': 0.57326

In [ ]:
vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True)
A = vec.fit_transform(train_tok)
B = vec.transform(dev_tok)

for c100 in range(100, 300, 25):
    c = c100 / 100
    multilabel[c] = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A, y_train)
    y_pred = multilabel[c].predict(B)
    print(f"=== C={c}, {c}")
    print(compute_metrics(y_dev, y_pred))

=== C=1.0, 1.0
{'accuracy': 0.43357820630547905, 'f1': 0.4639510064438256, 'precision': 0.4297905105335521, 'recall': 0.6102667049626519, 'hamming_loss': 0.030969871878962787}
=== C=1.25, 1.25
{'accuracy': 0.45454545454545453, 'f1': 0.4741469755451201, 'precision': 0.4488947523502235, 'recall': 0.6001223513153785, 'hamming_loss': 0.02954873409418864}
=== C=1.5, 1.5
{'accuracy': 0.465564738292011, 'f1': 0.47866905726333486, 'precision': 0.4564828616775423, 'recall': 0.5959096612613257, 'hamming_loss': 0.028657790021426384}
=== C=1.75, 1.75
{'accuracy': 0.47796143250688705, 'f1': 0.489565304352822, 'precision': 0.47838506882445003, 'recall': 0.5931420461969061, 'hamming_loss': 0.02784336875245966}
=== C=2.0, 2.0
{'accuracy': 0.48806244260789716, 'f1': 0.49213598038279266, 'precision': 0.48277473410530447, 'recall': 0.5889316842891587, 'hamming_loss': 0.02717106126197035}
=== C=2.25, 2.25
{'accuracy': 0.4989286807468626, 'f1': 0.497357754397144, 'precision': 0.49667765169345357, 'recall':

In [42]:
vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True)
A = vec.fit_transform(train_tok)
B = vec.transform(dev_tok)

for c100 in range(250, 275, 5):
    c = c100 / 100
    multilabel[c] = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A, y_train)
    y_pred = multilabel[c].predict(B)
    print(f"=== C={c}, {c}")
    print(compute_metrics(y_dev, y_pred))

=== C=2.5, 2.5
{'accuracy': 0.5071931435567799, 'f1': 0.500750922198366, 'precision': 0.5140615258117424, 'recall': 0.5790022862295928, 'hamming_loss': 0.026077878350605623}
=== C=2.55, 2.55
{'accuracy': 0.509641873278237, 'f1': 0.5037874182855876, 'precision': 0.5266271249278097, 'recall': 0.5786663224542475, 'hamming_loss': 0.025946696401241857}
=== C=2.6, 2.6
{'accuracy': 0.5113253749617386, 'f1': 0.5061169881979766, 'precision': 0.5514700010458129, 'recall': 0.5787049985127677, 'hamming_loss': 0.025842844024662207}
=== C=2.65, 2.65
{'accuracy': 0.512243648607285, 'f1': 0.5104979213073516, 'precision': 0.588563349393221, 'recall': 0.5784142118365505, 'hamming_loss': 0.02575538939175303}
=== C=2.7, 2.7
{'accuracy': 0.5139271502907866, 'f1': 0.5110570925513228, 'precision': 0.589402790042684, 'recall': 0.5780076089870085, 'hamming_loss': 0.025646071100616556}
